In [1]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import PCA
import numpy as np
from scipy.sparse import csr_matrix
import csv
import random
import pickle
import json
import pandas as pd
def data_process_user(event_file: str ):
    user_documents_dict = {}
    with open(event_file, "r", encoding="latin-1") as f:
        f.readline()
        count = 0
        for line in f:
            count += 1
            line = line.split("\t")
            times_plit = line[7].split(' ')
            if times_plit[1] == 'Apr' and times_plit[2] >= '10': #Apr03到Apr09是我需要的预处理数据TKY NYC 一样
                break
            if int(line[0]) not in user_documents_dict:
                user_documents_dict[int(line[0])] = [line[2]] # 用户去过的地点的类别
            else:
                user_documents_dict[int(line[0])].append(line[2])
        user_documents_list = []
        user_id_list =[]

        for key in user_documents_dict:
            user_id_list.append(key)
            tempstr = ' '.join(user_documents_dict[key])
            user_documents_list.append(tempstr)

    return user_id_list, user_documents_list, user_documents_dict

def time_process(time:list):
    hour = list(map(int, time[3].split(':')))[0]
    hour_shift = int(time[4]) / 60
    hour_true = hour + hour_shift
    if hour_true >= 8 and hour_true < 12:
        time_zone = "morning"
    elif hour_true >= 12 and hour_true < 14:
        time_zone = "noon"
    elif hour_true >= 14 and hour_true < 16:
        time_zone = "afternoon"
    elif hour_true >= 16 and hour_true < 22:
        time_zone = "night"
    else:
        time_zone = "rest"
    return  time_zone
def data_process_item(event_file: str ):
    item_documents = {}
    with open(event_file, "r", encoding="latin-1") as f:
        f.readline()
        count = 0
        for line in f:
            count += 1
            line = line.split("\t")
            times_plit = line[7].split(' ')
            if times_plit[1] == 'Apr' and times_plit[2] >= '10': #Apr03到Apr09是我需要的预处理数据
                break
            time_zone = time_process(times_plit)
            if line[2] not in item_documents:
                item_documents[line[2]] = [line[2]]# 地点的类别
                item_documents[line[2]].append(time_zone)
            else:
                item_documents[line[2]].append(line[2])
                item_documents[line[2]].append(time_zone)
        item_documents_list = []
        item_id_list =[]

        for key in item_documents:
            item_id_list.append(key)
            tempstr = ' '.join(item_documents[key])
            item_documents_list.append(tempstr)

    return item_id_list, item_documents_list

def data_process_event(event_file: str):
    user_id_list = []
    cat_id_list = []
    week_list = []
    time_list = []
    events = {}
    with open(event_file, "r", encoding="latin-1") as f:
        f.readline()
        count = 0
        for line in f:
            line = line.split("\t")
            time_split = line[7].split(' ')
            if time_split[1] == 'Apr' and time_split[2] < '10': #Apr03到Apr09是可观测数据,测试时不要
                count += 1
                continue

            time_zone = time_process(time_split)# 输出当前时间段
            time_list.append(time_zone)
            user_id_list.append(line[0])
            cat_id_list.append(line[2])
            week_list.append(time_split[0])

            events["user_ids"] = user_id_list
            events["cat_ids"] = cat_id_list
            events["week_periods"] = week_list
            events["time_periods"] = time_list

    return events

# def cat_id_map(event_file: str):
#     with open(event_file, "r", encoding="latin-1") as f:
#         f.readline()
#         cat_ids_list = []
#         cat_ids_dict = {}
#         for line in f:
#             line = line.split("\t")
#             cat_ids_list.append(line[2])
#         unique_cat_ids = list(set(cat_ids_list))
#         for i in range(len(unique_cat_ids)):
#             cat_ids_dict[unique_cat_ids[i]] = str(i) #id : 新id
#         # 将字典存储为文件
#         with open('./cat_id_map_dict.pkl', 'wb') as f:
#             pickle.dump(cat_ids_dict, f)



In [2]:
#数据读入预处理
event_file = r"./2 NYC and Tokyo Check-in Datase/dataset_TSMC2014_TKY.txt"

# #用户观测向量生成
# import os
# for dirname, _, filenames in os.walk('./dataset/foursquare/'):
#     for filename in filenames:
#         print(os.path.join(dirname, filename))

user_id_list, user_documents_list, user_documents_dict = data_process_user(event_file) # 返回前一周的user列表，元素是venue id

# # 初始化 TfidfVectorizer
# vectorizer = TfidfVectorizer()
# # 计算 TF-IDF
# tfidf_matrix = vectorizer.fit_transform(user_documents_list) # 每一行代表一个文章，每一列代表一个单词
# #将稀疏矩阵转化为密集矩阵再转化为array
# dense_matrix = np.asarray(csr_matrix.todense(tfidf_matrix))
# # 创建 PCA 对象，指定降维后的维度为 25
# pca = PCA(n_components=25)
# # 对数据矩阵进行拟合和转换
# X_projected = pca.fit_transform(dense_matrix)
# #将降维后的向量与user id 匹配
# filename = 'User_FeatureVectors.dat'
# with open(filename, 'w', newline='') as f:
#     for i in range(X_projected.shape[0]):
#         new_line = str(user_id_list[i]) + '\t' + ';'.join(str(X_projected[i, :]).strip('[').strip(']').split()) + '\n'
#         f.write(new_line)

# 读取没有列名的CSV文件
data = pd.read_csv("./TKY/POI_Glove_FeatureVectors.csv", header=0, sep=' ')
FeatureVectors = {}
for index, row in data.iterrows():
    key = int(row[0])
    value = row[1:].tolist()
    FeatureVectors[key] = np.array(value)
# cat_id_map_dict = {}
# with open("./dataset/foursquare/cat_id_map_dict.json", 'r') as f:
#     cat_id_map_dict = json.load(f)
with open("./TKY/cat_id_map_dict.json", 'r') as f:
    cat_id_map_dict = json.load(f)
users = []
userfeatures = []
for user in user_documents_dict:
    list = user_documents_dict[user]
    newlist =  [cat_id_map_dict.get(item) for item in list]
    featurelist = [FeatureVectors.get(item) for item in newlist]
    featuremean = np.mean(featurelist,axis = 0)
    users.append(user)
    userfeatures.append(featuremean)
users_glove_feature = pd.DataFrame({'uid': users, 'featuremean': userfeatures})
# 展开userfeatures列
users_glove_feature_expanded = users_glove_feature["featuremean"].apply(pd.Series)
users_glove_feature_expanded.columns = [f"u{i}" for i in range(len(users_glove_feature_expanded.columns))]
# 合并展开后的列和原始列
result_df = pd.concat([users_glove_feature.drop("featuremean", axis=1), users_glove_feature_expanded], axis=1)
result_df



,uid,u0,u1,u2,u3,u4,u5,u6,u7,u8,...,u20,u21,u22,u23,u24,u25,u26,u27,u28,u29
0,868,0.769533,-0.584054,0.033698,0.615681,0.616085,-0.113057,-0.286729,-0.122116,0.108261,...,-0.011484,-0.192883,0.242007,-0.202697,0.155682,0.173926,0.106891,0.214119,0.101837,-0.226468
1,114,-0.406192,-0.277822,1.644915,-0.310860,-0.180669,-0.281364,-0.953587,0.167760,0.033061,...,-0.516856,0.010219,0.712776,-0.321756,-0.266093,0.284349,0.414463,-0.451735,0.114916,-0.271819
2,1458,-1.182736,0.310900,0.544380,-0.876621,-0.101550,-1.087947,-0.943926,-0.251927,-0.334852,...,0.222816,-0.246735,0.161517,-0.173870,-0.450140,0.086370,0.176904,0.080288,0.044055,-0.200206
3,1541,-0.062119,-0.362071,0.062043,-0.134294,0.222651,0.123801,-0.277163,0.182900,0.157453,...,0.109930,0.012128,0.226406,0.049729,0.011248,-0.057104,0.065104,-0.126770,0.097899,0.051079
4,1635,-0.008183,0.158957,0.758697,0.242165,-1.188651,-0.053958,-0.416525,-0.167453,-0.193359,...,-0.712486,-0.190245,0.471916,0.026797,0.133422,0.364891,0.232172,-0.215924,-0.084001,-0.163277
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1587,902,-0.462251,0.745678,2.081868,-0.397688,-0.135593,-0.928747,-1.525605,-0.000869,-0.631042,...,-0.600546,0.081528,0.943777,-0.228986,-0.442254,0.369199,0.328832,-1.093267,0.249341,-0.593085
1588,1997,-0.462251,0.745678,2.081868,-0.397688,-0.135593,-0.928747,-1.525605,-0.000869,-0.631042,...,-0.600546,0.081528,0.943777,-0.228986,-0.442254,0.369199,0.328832,-1.093267,0.249341,-0.593085
1589,1626,0.901381,-0.165725,0.276448,0.221292,-1.378187,-0.430944,-0.451907,-0.189746,-0.075593,...,-0.161894,1.053431,-0.344790,0.489834,-0.986503,-0.523239,-0.808056,0.508041,-0.257871,0.249288
1590,1257,0.305584,0.578995,1.523103,-0.047897,-0.869017,-0.971269,-1.283481,0.526309,-0.154042,...,-0.488286,-0.008049,0.863411,0.524327,-0.254041,0.380667,0.021030,-0.803806,-0.134098,-0.861228


In [10]:
result_df.to_csv('./TKY/User_Glovemean_FeatureVectors.csv', sep=' ', index=False, header=True)

In [ ]:

# POI 类别id映射
   # cat_id_map(event_file)


# POI 类别观测向量生成

    # item_id_list, item_documents_list = data_process_item(event_file)
    # # 重新映射cat id
    # with open('./cat_id_map_dict.pkl', 'rb') as f:
    #     cat_id_map_dict = pickle.load(f)
    # for i in range(len(item_id_list)):
    #     item_id_list[i] = cat_id_map_dict[item_id_list[i]]
    # # 初始化 TfidfVectorizer
    # vectorizer = TfidfVectorizer()
    # # 计算 TF-IDF
    # tfidf_matrix = vectorizer.fit_transform(item_documents_list) # 每一行代表一个文章，每一列代表一个单词
    # #将稀疏矩阵转化为密集矩阵再转化为array
    # dense_matrix = np.asarray(csr_matrix.todense(tfidf_matrix))
    # # 创建 PCA 对象，指定降维后的维度为 25
    # pca = PCA(n_components=10)
    # # 对数据矩阵进行拟合和转换
    # X_projected = pca.fit_transform(dense_matrix)
    # #将降维后的向量与user id 匹配
    # filename = 'POI_Cat_FeatureVectors.dat'
    # with open(filename, 'w', newline='') as f:
    #     for i in range(X_projected.shape[0]):
    #         new_line = str(item_id_list[i]) + '\t' + ';'.join(str(X_projected[i, :]).strip('[').strip(']').split()) + '\n'
    #         f.write(new_line)



 #生成事件流包含
    # """
    #     events["user_ids"] = user_id_list
    #     events["cat_ids"] = cat_id_list
    #     events["week_periods"] = week_list
    #     events["time_periods"] = time_list
    # """
    # events = data_process_event(event_file)
    # #生成armpool
    # #给出cat(类别)字典
    # unique_cat_ids = list(set(events["cat_ids"]))
    #
    # #将catid从0重新映射
    # with open('./cat_id_map_dict.json', 'r') as f:
    #     cat_id_map_dict = json.load(f)
    # for i in range(len(events["cat_ids"])):
    #     events["cat_ids"][i] = cat_id_map_dict[events["cat_ids"][i]]
    # for i in range(len(unique_cat_ids)):
    #     unique_cat_ids[i] = cat_id_map_dict[unique_cat_ids[i]]
    #
    # #采样
    # num_samples = 24
    # # 从my_list中随机选择num_samples个元素，并从中排除exclude_list中的元素
    # armpool = []
    #
    # for i in range(len( events["user_ids"])):
    #     #随机选出armpool剩余元素，注意避开用户实际选择元素
    #     arms_list = random.sample([x for x in unique_cat_ids if x != events["cat_ids"][i]], num_samples)
    #     #添加用户选择元素到第一个位置
    #     arms_list.insert(0, events["cat_ids"][i])
    #     arms_list = list(map(str, arms_list))  # 使用map函数将int换为str
    #     armpool.append(arms_list)
    #
    # filename = './Events_NYC.dat'
    # with open(filename, 'w', newline='') as f:
    #     for i in range(len(events["user_ids"])):
    #         new_line = str(events["user_ids"][i]) + '\t\t' + ' '.join(armpool[i]) + '\t\t' + events["week_periods"][i] + '\t\t' + events["time_periods"][i] + '\n'
    #         f.write(new_line)







